<img style="float: left; margin: 30px 15px 15px 15px;" src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="400" height="600" /> 
    
    
## <font color='navy'> Teoría Pos-Moderna de Portafolios: Práctica
    
### <font color='navy'> Portafolios de Inversión

    Mtro. Sean Nicolás González Vázquez

---

### <font color='navy'> 0.- Introducción
    
En la sesión anterior, exploramos varios métodos de la **Teoría Posmoderna de Portafolios (TPMP)**, que se enfoca en la optimización de portafolios separando claramente el riesgo en dos componentes: el downside y el upside risk. 
    
A grandes rasgos, esta teoría amplía la clásica Teoría Moderna de Portafolios, al poner énfasis en los riesgos que afectan negativamente las inversiones, y aquellos que representan oportunidades de ganancia.

La principal contribución de este enfoque radica en su capacidad para descomponer el riesgo en dos categorías clave:

+ **Downside risk**: Riesgo de incurrir en pérdidas, es decir, cuando los rendimientos caen por debajo de un umbral establecido, como podría ser un benchmark o simplemente rendimientos negativos.   
    
    
+ **Upside risk**: Riesgo (potencial) de obtener ganancias.   
    
    
+ Mientras que el downside implica posibles pérdidas para el portafolio, el upside representa las oportunidades de obtener rendimientos positivos.

Dentro de esta estructura, profundizamos en **dos métodos de Asset Allocation** basados en esta separación de riesgos: el método de la **Mínima Semivarianza** y el método de **Máximo Omega**. 
    
Ambos enfoques proporcionan alternativas interesantes a los modelos tradicionales propuestos por Markowitz y Sharpe.   
    
El primero, la Mínima Semivarianza, es una extensión del clásico método de Mínima Varianza, ya que se centra exclusivamente en minimizar el riesgo a la baja (downside risk), ignorando las fluctuaciones al alza. El segundo, el Máximo Omega, es una alternativa al Ratio de Sharpe, con la diferencia clave de que se enfoca en maximizar la relación entre las ganancias potenciales (upside) y las pérdidas posibles (downside), proporcionando una evaluación más matizada del rendimiento ajustado al riesgo.  

> Con este contexto, implementaremos ambos métodos, utilizando optimización montecarlo.

---

### <font color='navy'> 1.- Descarga de Datos y Obtención de Métricas

En este ejercicio, vamos a construir un portafolio compuesto por siete activos financieros previamente seleccionados:

+ PG: Procter & Gamble
+ COST: Costco Wholesale
+ KO: Coca-Cola
+ WMT: Walmart
+ CLX: Clorox
+ K: Kellogg's
+ KHC: Kraft Heinz

Como puedes observar, todos estos activos pertenecen a compañías que producen bienes de consumo básico, tales como productos de limpieza, alimentos y bebidas, entre otros. 


Estos activos forman parte de un sector específico conocido como Consumer Staples. Este sector es considerado anticíclico por naturaleza. Por ello, se le clasifica como un sector defensivo, ideal para inversiónistas que buscan estabilidad y son aversos al riesgo.

> **Consumer Staples:** Hace referencia a bienes de consumo necesarios para la vida diaria, como alimentos, bebidas, productos de higiene personal y del hogar. Estos productos suelen ser demandados de forma constante, independientemente del ciclo económico.  

> **Sector Defensivo:** Se refiere a aquellos sectores de la economía cuyos activos tienden a ser menos volátiles y más resilientes en tiempos de crisis o desaceleraciones económicas. Las empresas en estos sectores producen bienes y servicios esenciales, lo que les permite mantener una demanda relativamente estable incluso en condiciones adversas del mercado.


Como gestor del portafolio (portfolio manager), tu objetivo es encontrar las ponderaciones óptimas que minimicen la semivarianza del portafolio y maximicen el ratio de Omega.

In [50]:
# Importación de Librerías
import numpy as np
import pandas as pd
import yfinance as yf

np.set_printoptions(suppress=True, precision=6)

In [2]:
# Descarga de Datos
prices=yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'], 
                    start='2020-01-01', end='2026-03-06')['Close']

/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  6 of 6 completed


In [3]:
prices.head()

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2020-01-02,126.064590,266.497467,23.362871,45.745106,105.524033,36.433834
2020-01-03,126.312401,266.716980,23.089411,45.495541,104.814331,36.112194
2020-01-06,126.692390,266.790100,23.141142,45.478901,104.959663,36.038677
2020-01-07,125.164185,266.369598,22.734640,45.129520,104.309830,35.704792
2020-01-08,125.535912,269.423248,22.608990,45.212700,104.754456,35.582256


In [4]:
# Calculo de rendimientos
returns = prices.pct_change().dropna()

In [5]:
# Calcular rendimiento promedio
assets_mean_return = returns.mean()
assets_mean_return

Ticker
CLX     0.000073
COST    0.000954
KHC     0.000155
KO      0.000418
PG      0.000328
WMT     0.000893
dtype: float64

In [6]:
# Calcular matriz de correlación
assets_corr = returns.corr()
assets_corr

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,1.000000,0.326487,0.376578,0.334535,0.506711,0.312927
COST,0.326487,1.000000,0.307689,0.424536,0.483058,0.611316
KHC,0.376578,0.307689,1.000000,0.551005,0.535309,0.319343
KO,0.334535,0.424536,0.551005,1.000000,0.646583,0.370351
PG,0.506711,0.483058,0.535309,0.646583,1.000000,0.476900
WMT,0.312927,0.611316,0.319343,0.370351,0.476900,1.000000


---

### <font color='navy'> 2.- Portafolio Eficiente en Mínima Semivarianza  
    
  
### <font color='navy'>  **Mínima Semivarianza**

Método de la optimización de portafolios que busca minimizar el potencial riesgo de pérdida (*downside risk*) de un portafolio de inversión. ´


**Separar los rendimientos por debajo de 0 (pérdidas), las ganancias convertirlas en cero.**

$$R_{below, i} = min(r_i, 0)$$


In [7]:
below_zero = returns[returns < 0].fillna(0)
below_zero.head()

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2020-01-03,0.000000,0.000000,-0.011705,-0.005456,-0.006725,-0.008828
2020-01-06,0.000000,0.000000,0.000000,-0.000366,0.000000,-0.002036
2020-01-07,-0.012062,-0.001576,-0.017566,-0.007682,-0.006191,-0.009265
2020-01-08,0.000000,0.000000,-0.005527,0.000000,0.000000,-0.003432
2020-01-09,0.000000,0.000000,-0.000327,0.000000,0.000000,0.000000


**Calcular downside risk para cada activo.**

$$\sigma_{d, i} = \sigma(R_{below, i})$$

In [8]:
# Calcular downside risk
downside_risk = np.array(below_zero.std())
downside_risk

array([0.01062705, 0.00905878, 0.01060862, 0.0081278 , 0.00817945,
       0.00873361])

**Calcular matriz de Semivarianza.**

Antes de encontrar la matriz de semivarianza, simplifiquemos el cálculo......

Recordando, la varianza de dos activos financieros si tenemos la correlación entre ambos y sus desviaciones estandár individuales esta dada por $\sigma^2_{i, j} = \rho_{i, j} \sigma_i \sigma_j$. Para calcular la semi-varianza entre dos activos financieros utilizamos la misma fórmula, simplemente cambiando la desviación estandár por la semi-desviación (el *downside risk*), es decir, la semivarianza para dos activos financieros esta dada por $\sigma^2_{d(i, j)} = \rho_{i, j} \sigma_{(d, i)} \sigma_{(d, j)}$.

Esto debe hacerse iterativamente hasta construir la matriz de semivarianza, como vimos en la clase pasada. 

Para simplificar el ejercicio, podemos tomar el vector de $nx1$ de los downside risk individuales:

$$\sigma_d = \begin{bmatrix}
\sigma_{d, 1} \\
\sigma_{d, 2} \\
\vdots \\
\sigma_{d, n}
\end{bmatrix}$$

Y multiplicarla matricialmente por su transpuesta:

$$\sigma_d \sigma_d^T = 
\begin{bmatrix}
\sigma_{d, 1} \\
\sigma_{d, 2} \\
\vdots \\
\sigma_{d, n}
\end{bmatrix}
\begin{bmatrix}
\sigma_{d, 1} & \sigma_{d, 2} & \cdots & \sigma_{d, n}
\end{bmatrix}
=
\begin{bmatrix}
\sigma_{d, 1}^2 & \sigma_{d, 1} \sigma_{d, 2} & \cdots & \sigma_{d, 1} \sigma_{d, n} \\
\sigma_{d, 2} \sigma_{d, 1} & \sigma_{d, 2}^2 & \cdots & \sigma_{d, 2} \sigma_{d, n} \\
\vdots & \vdots & \ddots & \vdots \\
\sigma_{d, n} \sigma_{d, 1} & \sigma_{d, n} \sigma_{d, 2} & \cdots & \sigma_{d, n}^2
\end{bmatrix}$$

Notesé que esto equivale a multiplicar el elemento $i$ con el $j$ de la matriz de semivarianza, donde la diagonal son los downside al cuadrado (puedes utilizar la función de `np.multiply`). Finalmente realizamos el producto Hadamard de esta matriz con la matriz de correlación y con esto obtenemos la matriz de semivarianza.

$$S = \begin{bmatrix}
1 & \rho_{12} & \cdots & \rho_{1n} \\
\rho_{21} & 1 & \cdots & \rho_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
\rho_{n1} & \rho_{n2} & \cdots & 1
\end{bmatrix} \circ \begin{bmatrix}
\sigma_{d, 1}^2 & \sigma_{d, 1} \sigma_{d, 2} & \cdots & \sigma_{d, 1} \sigma_{d, n} \\
\sigma_{d, 2} \sigma_{d, 1} & \sigma_{d, 2}^2 & \cdots & \sigma_{d, 2} \sigma_{d, n} \\
\vdots & \vdots & \ddots & \vdots \\
\sigma_{d, n} \sigma_{d, 1} & \sigma_{d, n} \sigma_{d, 2} & \cdots & \sigma_{d, n}^2
\end{bmatrix}$$

El producto [Hadamard](https://en.wikipedia.org/wiki/Hadamard_product_(matrices)#:~:text=In%20mathematics,%20the%20Hadamard%20product%20also%20known%20as%20the) es una multiplicación elemento a elemento de cada matriz según su posición específica (para realizarla en python, puedes utilizar simplimente `*`). Al realizar esta operación obtenemos la matriz de semivarianza para el portafolio:

$$S=
\begin{bmatrix}
\sigma_{d, 1}^2 & \rho_{12} \sigma_{d, 1} \sigma_{d, 2} & \cdots & \rho_{1n} \sigma_{d, 1} \sigma_{d, n} \\
\rho_{21} \sigma_{d, 2} \sigma_{d, 1} & \sigma_{d, 2}^2 & \cdots & \rho_{2n} \sigma_{d, 2} \sigma_{d, n} \\
\vdots & \vdots & \ddots & \vdots \\
\rho_{n1} \sigma_{d, n} \sigma_{d, 1} & \rho_{n2} \sigma_{d, n} \sigma_{d, 2} & \cdots & \sigma_{d, n}^2
\end{bmatrix}$$


**Calcular la matriz de semivarianza del portafolio.**

Con la matriz de semivarianza, podemos obtener la semivarianza de un portafolio de inversión con la fórmula:

$$s = w^T S w$$

In [9]:
semivar_matrix = downside_risk.reshape(len(returns.columns), 1) @ downside_risk.reshape(1, len(returns.columns)) * assets_corr
semivar_matrix

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,0.000113,0.000031,0.000042,0.000029,0.000044,0.000029
COST,0.000031,0.000082,0.000030,0.000031,0.000036,0.000048
KHC,0.000042,0.000030,0.000113,0.000048,0.000046,0.000030
KO,0.000029,0.000031,0.000048,0.000066,0.000043,0.000026
PG,0.000044,0.000036,0.000046,0.000043,0.000067,0.000034
WMT,0.000029,0.000048,0.000030,0.000026,0.000034,0.000076


**Minimizar semivarianza cambiando ponderaciones.**

$$min_w \hspace{0.5cm} \sigma_d^2 = w^T S w$$
    
$$s.a. \hspace{0.5cm} \sum_{i=1}^n w_i = 1$$
 
$$\hspace{0.8cm} w_i > 0 $$



In [10]:
# Simular portafolios con montecarlo
np.random.seed(21)
n_sims = 100_000

w_aleatorio = np.random.dirichlet(np.ones(len(returns.columns)), size=n_sims)

semivar = [w.T @ semivar_matrix @ w for w in w_aleatorio]
semivar

[np.float64(7.0326289705161e-05),
 np.float64(4.696563578931553e-05),
 np.float64(4.5945085907822944e-05),
 np.float64(4.484495969185724e-05),
 np.float64(5.572891789033419e-05),
 np.float64(4.6232618075931257e-05),
 np.float64(5.1021966512003174e-05),
 np.float64(4.751448704887696e-05),
 np.float64(4.678960746260843e-05),
 np.float64(4.7894210737793744e-05),
 np.float64(4.802328882241924e-05),
 np.float64(5.3822070185484487e-05),
 np.float64(5.053880224794858e-05),
 np.float64(6.061645304604062e-05),
 np.float64(4.8596572627384997e-05),
 np.float64(4.7826033414827504e-05),
 np.float64(4.773119980118494e-05),
 np.float64(4.861467555126936e-05),
 np.float64(5.846724170849038e-05),
 np.float64(4.99815139886819e-05),
 np.float64(4.641975710837236e-05),
 np.float64(5.070598605561316e-05),
 np.float64(5.311663277365508e-05),
 np.float64(4.816256893735101e-05),
 np.float64(5.512531151618529e-05),
 np.float64(4.599906505726498e-05),
 np.float64(5.285739141729875e-05),
 np.float64(4.6489717802

In [11]:
# Obtener la mínima semivarianza
min_semivar = min(semivar)
min_semivar

np.float64(4.3022171008343976e-05)

In [12]:
# Obtener los pesos del portafolio con mínima semivarianza
w_min_semivar = w_aleatorio[np.argmin(semivar)]

dict(zip(returns.columns, w_min_semivar*100))

{'CLX': np.float64(11.78043383741072),
 'COST': np.float64(13.537089029719343),
 'KHC': np.float64(3.629697154565678),
 'KO': np.float64(33.78668653192883),
 'PG': np.float64(11.008998614021582),
 'WMT': np.float64(26.25709483235384)}

---

### <font color='navy'> 3.- Portafolio Eficiente en Máximo Omega
    
    
Portafolio que maximiza el ratio omega de un portafolio de inversión, el ratio Omega mide la relación entre el upside risk (potencial de ganancias) y el downside risk (potencial de pérdidas) de un activo financiero.

**Separar los rendimientos en dos, aquellos por debajo de cero y los que están por encima de cero.**

$$R_{below, i} = min(r_i, 0)$$

$$R_{beneath, i} = max(r_i, 0)$$

In [26]:
below_zero = returns[returns < 0].fillna(0)
above_zero = returns[returns > 0].fillna(0)

**Calcular downside y upside risk para cada activo.**

$$\sigma_{d, i} = \sigma(R_{below, i})$$

$$\sigma_{u, i} = \sigma(R_{beneath, i})$$

In [27]:
upside = above_zero.std()
downside = below_zero.std()

In [30]:
upside * 100

Ticker
CLX     1.000773
COST    0.932918
KHC     1.046380
KO      0.771651
PG      0.798164
WMT     0.951844
dtype: float64

In [31]:
downside * 100

Ticker
CLX     1.062705
COST    0.905878
KHC     1.060862
KO      0.812780
PG      0.817945
WMT     0.873361
dtype: float64

**Calcular Omega individual para cada activo.**

$$\Omega_i = \frac{\sigma_{u, i}}{\sigma_{d, i}}$$


In [34]:
omegas = upside/downside

**Calcular Omega del portafolio y maximizarla cambiando ponderaciones.**

$$max_w \hspace{0.5cm} \Omega_p = \sum_{i=1}^{n} \Omega_i * w_i$$
    
$$s.a. \hspace{0.5cm} \sum_{i=1}^n w_i = 1$$
 
$$\hspace{0.8cm} w_i > 0 $$

In [51]:
# Simular portafolios con montecarlo
np.random.seed(30)
n_size = 1_000_000

w_aleatorio = np.random.dirichlet(np.ones(len(returns.columns)), size = n_size)

omegas_aleatorio = [np.dot(w, omegas) for w in w_aleatorio]

In [ ]:
# Obtener el maximo omega
np.argmax(omegas_aleatorio)

np.int64(558033)

In [54]:
# Obtener los pesos del portafolio con maximo omega
result = w_aleatorio[np.argmax(omegas_aleatorio)]
np.round(result, 6)

array([0.020678, 0.000997, 0.032488, 0.000143, 0.014705, 0.930989])

---

### 4.- Target Minimum Semivariance Portfolio
    

Ahora, supón que deseas optimizar las ponderaciones del portafolio considerando aquellos activos que tienen mayor probabilidad de superar a tu *benchmark*. Para ello, emplearás una estrategia basada en la *target semivariance*.

En este caso, asume que tu *benchmark* es el ETF `KXI`, que sigue el sector de bienes de consumo global. 
    
El objetivo es encontrar las ponderaciones óptimas que minimicen el riesgo de no superar el rendimiento de `KXI`. 
    
    
Recuerda que este enfoque ajustado al *benchmark* permite identificar aquellos activos que tienen un mejor desempeño relativo y, por lo tanto, una mayor probabilidad de generar *alpha*. Al minimizar la semivarianza relativa al *benchmark*, estarás optimizando el portafolio para que minimice el riesgo de rendimientos inferiores al de `KXI`.

Recuerda que los pasos son reiterativos y siguen la misma estructura que el método de mínima semivarianza, solo que ahora restarás los rendimientos al rendimiento del *benchmark* a los de tus activos.

In [56]:
# Descarga de Precios del Benchmark
benchmark = yf.download("KXI", start='2020-01-01', end='2026-03-06')["Close"]
benchmark.head(5)

/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  1 of 1 completed


Ticker,KXI
Date,
2020-01-02,47.864311
2020-01-03,47.821049
2020-01-06,47.968128
2020-01-07,47.656651
2020-01-08,47.647999


In [58]:
# Obtener rendimientos del benchmark
rets_benchmark = benchmark.pct_change().dropna()
rets_benchmark.head(5)

Ticker,KXI
Date,
2020-01-03,-0.000904
2020-01-06,0.003076
2020-01-07,-0.006493
2020-01-08,-0.000182
2020-01-09,0.004721


In [59]:
# Sustraer rendimiento del benchmark de rendimientos individuales
diffs = returns - rets_benchmark.values
diffs

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2020-01-03,0.002870,0.001728,-0.010801,-0.004552,-0.005822,-0.007924
2020-01-06,-0.000067,-0.002801,-0.000835,-0.003441,-0.001689,-0.005111
2020-01-07,-0.005569,0.004917,-0.011073,-0.001189,0.000302,-0.002771
2020-01-08,0.003151,0.011646,-0.005345,0.002025,0.004444,-0.003250
2020-01-09,0.000741,0.011329,-0.005048,0.013494,0.006217,0.005609
...,...,...,...,...,...,...
2026-02-27,-0.009423,0.012822,-0.009923,0.001617,0.009518,0.016821
2026-03-02,0.015056,0.009874,0.013745,0.001379,-0.004261,0.011165
2026-03-03,-0.028855,0.020765,0.003947,0.004809,-0.007400,0.022152


In [61]:
# Separar los rendimientos por debajo de cero
bz = diffs[diffs < 0].fillna(0)

In [ ]:
# Calcular target downside risk
target_downside = np.array(bz.std())


Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,0.000089,0.000022,0.000029,0.000016,0.000025,0.000022
COST,0.000022,0.000051,0.000018,0.000015,0.000018,0.000033
KHC,0.000029,0.000018,0.000066,0.000022,0.000022,0.000019
KO,0.000016,0.000015,0.000022,0.000024,0.000016,0.000014
PG,0.000025,0.000018,0.000022,0.000016,0.000026,0.000018
WMT,0.000022,0.000033,0.000019,0.000014,0.000018,0.000055


In [ ]:
# Calcular la matriz de target semivariance
target_semivar_matriz = target_downside.reshape(len(returns.columns), 1) @ target_downside.reshape(1, len(returns.columns)) * returns.corr()
target_semivar_matriz

In [ ]:
# Simular portafolios con montecarlo
targets = [w.T @ target_semivar_matriz @ w for w in w_aleatorio]

array([0.014436, 0.059756, 0.004049, 0.494697, 0.339955, 0.087107])

In [ ]:
# Obtener la mínima target semivariance
w_semivar_target = w_aleatorio[np.argmin(targets)]
w_semivar_target

{'CLX': np.float64(0.0144),
 'COST': np.float64(0.0598),
 'KHC': np.float64(0.004),
 'KO': np.float64(0.4947),
 'PG': np.float64(0.34),
 'WMT': np.float64(0.0871)}

In [83]:
# Obtener los pesos del portafolio con mínima semivarianza target
dict(zip(returns.columns, np.round(w_semivar_target, 4)*100))

{'CLX': np.float64(13.04),
 'COST': np.float64(14.21),
 'KHC': np.float64(4.49),
 'KO': np.float64(33.12),
 'PG': np.float64(10.31),
 'WMT': np.float64(24.84)}

In [74]:
dict(zip(returns.columns, w_min_semivar*100))

{'CLX': np.float64(11.78043383741072),
 'COST': np.float64(13.537089029719343),
 'KHC': np.float64(3.629697154565678),
 'KO': np.float64(33.78668653192883),
 'PG': np.float64(11.008998614021582),
 'WMT': np.float64(26.25709483235384)}

    ¡Excelente! Ahora comparemos las ponderaciones obtenidas en el método de semivarianza normal con target semivariance, responde: ¿Hay diferencias? ¿Como las interpretas?

---

### <font color='navy'> 5.- Reflexión

> **El método de la semivarianza y el ratio omega** son dos alternativas interesantes para la optimización de portafolios de inversión, ya que **replantean el concepto tradicional del riesgo financiero**.

> **¿Cuál es mejor?** No hay una respuesta definitiva; **la elección** del método "adecuado" **depende de varios factores**, como el perfil del inversionista, el comportamiento histórico de los activos seleccionados (que puede hacer que un método funcione mejor que otro) y las restricciones u objetivos específicos del gestor.

> Es importante resaltar nuevamente que estos **métodos solo son tan efectivos como lo sean los activos incluidos en el portafolio**.

> Hasta ahora, no hemos comparado las distintas estrategias de asignación de activos, sino que hemos enfocado en cómo determinar los pesos óptimos. **En las próximas sesiones, nos centraremos en la comparación y selección de estrategias de inversión**. Esto te permitirá, con un conjunto de activos que conforman un portafolio, evaluar sus características y elegir la mejor estrategia basándote en las métricas de los múltiples portafolios óptimos.

---

### <font color='navy'> Tarea 5

En equipos de dos integrantes, respondan los siguientes puntos y entreguen un archivo `html` con sus resultados. No olviden incluir los nombres de los integrantes en el archivo:

**Punto a)** Implementen tres funciones que optimicen portafolios utilizando: mínima semivarianza, ratio omega y semivarianza objetivo, empleando la función `scipy.optimize.minimize`.

**Punto b)** Construyan un portafolio compuesto por 5 activos financieros y optimícenlo con los tres métodos. Utilicen las funciones desarrolladas en el punto a) para realizar la optimización. Recuerden que, para el portafolio con semivarianza objetivo, deberán seleccionar un benchmark adecuado y consistente con los activos elegidos. Como respuesta a este punto, se esperan las ponderaciones eficientes y el valor óptimo de la función objetivo para cada método de Asset Allocation.

**Punto c)** Escriban una breve conclusión sobre las ventajas y desventajas que observan entre los métodos TMP y TPMP.

**Extra:** El equipo que logre la menor semivarianza global en su portafolio optimizado por mínima semivarianza recibirá 10 puntos extra para el segundo examen parcial. La selección de activos con una buena relación entre upside y downside será clave para obtener estos puntos.



In [75]:
diffs = returns - 0.05/252
diffs

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2020-01-03,0.001767,0.000625,-0.011903,-0.005654,-0.006924,-0.009026
2020-01-06,0.002810,0.000076,0.002042,-0.000564,0.001188,-0.002234
2020-01-07,-0.012261,-0.001775,-0.017765,-0.007881,-0.006390,-0.009463
2020-01-08,0.002772,0.011266,-0.005725,0.001645,0.004064,-0.003630
2020-01-09,0.005263,0.015852,-0.000525,0.018017,0.010739,0.010132
...,...,...,...,...,...,...
2026-02-27,0.001929,0.024175,0.001430,0.012969,0.020870,0.028173
2026-03-02,-0.002951,-0.008133,-0.004262,-0.016628,-0.022268,-0.006842
2026-03-03,-0.044832,0.004788,-0.012030,-0.011168,-0.023377,0.006175


In [76]:
bz = diffs[diffs < 0].fillna(0)

In [77]:
# Calcular target downside risk
target_downside = np.array(bz.std())

In [78]:
# Calcular la matriz de target semivariance
target_semivar_matriz = target_downside.reshape(len(returns.columns), 1) @ target_downside.reshape(1, len(returns.columns)) * returns.corr()
target_semivar_matriz

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,0.000114,0.000032,0.000043,0.000029,0.000045,0.000029
COST,0.000032,0.000083,0.000030,0.000032,0.000036,0.000049
KHC,0.000043,0.000030,0.000114,0.000048,0.000047,0.000030
KO,0.000029,0.000032,0.000048,0.000067,0.000044,0.000027
PG,0.000045,0.000036,0.000047,0.000044,0.000068,0.000035
WMT,0.000029,0.000049,0.000030,0.000027,0.000035,0.000077


In [79]:
# Simular portafolios con montecarlo
targets = [w.T @ target_semivar_matriz @ w for w in w_aleatorio]

In [80]:
# Obtener la mínima target semivariance
w_semivar_target = w_aleatorio[np.argmin(targets)]
w_semivar_target

array([0.130364, 0.14206 , 0.044891, 0.331208, 0.103067, 0.24841 ])

In [82]:
dict(zip(returns.columns, np.round(w_semivar_target, 4)*100))

{'CLX': np.float64(13.04),
 'COST': np.float64(14.21),
 'KHC': np.float64(4.49),
 'KO': np.float64(33.12),
 'PG': np.float64(10.31),
 'WMT': np.float64(24.84)}